# Channel Revenue Summary

Reads `commerce.store_sales`, `commerce.catalog_sales`, and
`commerce.web_sales` (see `data-generators/tpcds_retail_ingest.ipynb`)
and computes omnichannel revenue and profit-margin performance by sales
channel and year — which channel (store, catalog, web) is actually
driving revenue and margin, and how much of it is promotion-driven.

Each channel's fact table uses its own column prefix (`ss_`, `cs_`,
`ws_`) for the same conceptual fields, so the first step standardizes
them to common column names and a literal `channel` label before
`unionByName`-ing the three into one fact frame, then joins in
`date_dim` (for year) and `promotion` (to flag promotional line items).

Re-runnable: `mode("overwrite")`, so re-running just refreshes the
table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "commerce"
TARGET_TABLE = "channel_revenue_summary_v1"

## Load raw tables

In [ ]:
store_sales_df = spark.table(f"{DATABASE_NAME}.store_sales")
catalog_sales_df = spark.table(f"{DATABASE_NAME}.catalog_sales")
web_sales_df = spark.table(f"{DATABASE_NAME}.web_sales")
date_dim_df = spark.table(f"{DATABASE_NAME}.date_dim")
promotion_df = spark.table(f"{DATABASE_NAME}.promotion")

## Standardize and union the three sales channels

In [ ]:
def to_common_shape(df, prefix, channel):
    return df.select(
        F.col(f"{prefix}_sold_date_sk").alias("sold_date_sk"),
        F.col(f"{prefix}_promo_sk").alias("promo_sk"),
        F.col(f"{prefix}_ext_sales_price").alias("ext_sales_price"),
        F.col(f"{prefix}_net_profit").alias("net_profit"),
        F.lit(channel).alias("channel"),
    )

sales_df = (
    to_common_shape(store_sales_df, "ss", "store")
    .unionByName(to_common_shape(catalog_sales_df, "cs", "catalog"))
    .unionByName(to_common_shape(web_sales_df, "ws", "web"))
)

## Join with date and promotion dimensions

In [ ]:
joined_df = (
    sales_df
    .join(
        date_dim_df.select("d_date_sk", "d_year"),
        sales_df.sold_date_sk == date_dim_df.d_date_sk,
        "left",
    )
    .join(
        promotion_df.select("p_promo_sk"),
        sales_df.promo_sk == promotion_df.p_promo_sk,
        "left",
    )
    .withColumn("is_promotional", F.col("p_promo_sk").isNotNull())
)

## Aggregate by channel and year

In [ ]:
summary_df = (
    joined_df
    .groupBy("channel", "d_year")
    .agg(
        F.count("*").alias("line_item_count"),
        F.round(F.sum("ext_sales_price"), 2).alias("total_revenue"),
        F.round(F.sum("net_profit"), 2).alias("total_profit"),
        F.round(
            F.sum(F.when(F.col("is_promotional"), F.col("ext_sales_price")).otherwise(0.0)),
            2,
        ).alias("promo_revenue"),
    )
    .withColumn(
        "profit_margin_pct",
        F.round(F.col("total_profit") * 100.0 / F.col("total_revenue"), 1),
    )
    .withColumn(
        "promo_revenue_share_pct",
        F.round(F.col("promo_revenue") * 100.0 / F.col("total_revenue"), 1),
    )
    .drop("promo_revenue")
    .orderBy("d_year", "channel")
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_rows FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY d_year, channel").show(30)